In [1]:
!pip install ccxt

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 152.5/152.5 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.5/6.5 MB 78.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 73.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 223.8/223.8 kB 15.2 MB/s eta 0:00:00


In [2]:
import ccxt

# 1. 初始化多个交易所
exchanges = {
    'Binance': ccxt.binance(),
    'OKX': ccxt.okx(),
    'Bybit': ccxt.bybit(),
    'Coinbase': ccxt.coinbase(),
}

symbol = 'BTC/USDT'

print(f"{'='*60}")
print(f"  BTC/USDT 多交易所对比 — {symbol}")
print(f"{'='*60}")

for name, ex in exchanges.items():
    try:
        # 获取订单簿（前10档）
        ob = ex.fetch_order_book(symbol, limit=10)
        bids = ob['bids']
        asks = ob['asks']

        if not bids or not asks:
            print(f"\n{name}: 数据为空")
            continue

        # 最优买卖价
        best_bid = bids[0][0]
        best_ask = asks[0][0]
        bid_vol = bids[0][1]
        ask_vol = asks[0][1]

        # 价差（基点）
        spread_bp = (best_ask - best_bid) / best_bid * 10000

        # 订单簿深度——前5档累计
        bid_depth_5 = sum(b[1] for b in bids[:5])
        ask_depth_5 = sum(a[1] for a in asks[:5])

        # 买卖不平衡比
        ratio = bid_depth_5 / ask_depth_5 if ask_depth_5 > 0 else 0

        print(f"\n{'─'*50}")
        print(f"  📍 {name}")
        print(f"  Best Bid: ${best_bid:,.2f}  x {bid_vol:.4f} BTC")
        print(f"  Best Ask: ${best_ask:,.2f}  x {ask_vol:.4f} BTC")
        print(f"  🟢 价差: {spread_bp:.2f} bp")
        print(f"  买方深度(前5档): {bid_depth_5:.4f} BTC")
        print(f"  卖方深度(前5档): {ask_depth_5:.4f} BTC")
        print(f"  买卖比: {ratio:.2f}")

    except Exception as e:
        print(f"\n  {name}: 错误 — {str(e)[:50]}")

print(f"\n{'='*60}")
print("  流动性排名猜想: Binance价差最小、深度最大")
print("  Coinbase可能深度较浅（美国合规成本高）")
print("  如果BTC/USDT价差突然>10bp -> 流动性预警！")
print(f"{'='*60}")

  BTC/USDT 多交易所对比 — BTC/USDT

  Binance: 错误 — binance GET https://api.binance.com/api/v3/exchang

──────────────────────────────────────────────────
  📍 OKX
  Best Bid: $60,095.50  x 0.3260 BTC
  Best Ask: $60,095.60  x 1.8317 BTC
  🟢 价差: 0.02 bp
  买方深度(前5档): 0.8023 BTC
  卖方深度(前5档): 3.4784 BTC
  买卖比: 0.23

  Bybit: 错误 — bybit GET https://api.bybit.com/v5/market/instrume

──────────────────────────────────────────────────
  📍 Coinbase
  Best Bid: $60,088.60  x 0.0158 BTC
  Best Ask: $60,096.79  x 0.0073 BTC
  🟢 价差: 1.36 bp
  买方深度(前5档): 0.1495 BTC
  卖方深度(前5档): 0.0655 BTC
  买卖比: 2.28

  流动性排名猜想: Binance价差最小、深度最大
  Coinbase可能深度较浅（美国合规成本高）
  如果BTC/USDT价差突然>10bp -> 流动性预警！


In [3]:
import ccxt
import pandas as pd
from datetime import datetime

# 连接 Coinbase
coinbase = ccxt.coinbase()
coinbase.load_markets()
print("✅ Coinbase 已连接，支持", len(coinbase.symbols), "个交易对")

✅ Coinbase 已连接，支持 1180 个交易对


In [4]:
# BTC/USDT 订单簿
symbol = 'BTC/USDT'
ob = coinbase.fetch_order_book(symbol)

bids = ob['bids']  # 买单
asks = ob['asks']  # 卖单

best_bid = bids[0][0]  # 买一价
best_ask = asks[0][0]  # 卖一价
spread = best_ask - best_bid
spread_pct = (spread / best_bid) * 100

print(f"📊 BTC/USDT 订单簿 @ {datetime.now().strftime('%H:%M:%S')}")
print(f"━━━━━━━━━━━━━━━━━━━━━━")
print(f"🟢 买一价: ${best_bid:,.2f}")
print(f"🔴 卖一价: ${best_ask:,.2f}")
print(f"📏 价差:   ${spread:.2f} ({spread_pct:.4f}%)")
print()
print(f"📋 前5档深度:")
print(f"  买盘:")
for i in range(5):
    print(f"   ${bids[i][0]:,.2f} × {bids[i][1]:.4f} BTC")
print(f"  卖盘:")
for i in range(5):
    print(f"   ${asks[i][0]:,.2f} × {asks[i][1]:.4f} BTC")

📊 BTC/USDT 订单簿 @ 06:51:52
━━━━━━━━━━━━━━━━━━━━━━
🟢 买一价: $60,120.54
🔴 卖一价: $60,131.11
📏 价差:   $10.57 (0.0176%)

📋 前5档深度:
  买盘:
   $60,120.54 × 0.0143 BTC
   $60,120.45 × 0.0158 BTC
   $60,120.44 × 0.0065 BTC
   $60,120.42 × 0.0017 BTC
   $60,119.21 × 0.0017 BTC
  卖盘:
   $60,131.11 × 0.0158 BTC
   $60,131.12 × 0.1180 BTC
   $60,132.11 × 0.0123 BTC
   $60,132.33 × 0.1180 BTC
   $60,136.84 × 0.0050 BTC


In [5]:
# 先看看哪些交易所 Colab 能连上
exchanges_names = ['coinbase', 'kraken', 'gemini']
working = {}

for name in exchanges_names:
    try:
        ex = getattr(ccxt, name)()
        ex.load_markets()
        ticker = ex.fetch_ticker('BTC/USDT')
        working[name] = ticker['last']
        print(f"✅ {name:10s} → ${ticker['last']:,.2f}")
    except Exception as e:
        print(f"❌ {name:10s} → 连不上 ({str(e)[:30]})")

print()
print("━━━━━━━━━━━━━━━━━━━━━━")
if len(working) >= 2:
    prices = list(working.values())
    diff = max(prices) - min(prices)
    diff_pct = (diff / min(prices)) * 100
    print(f"📏 最大价差: ${diff:.2f} ({diff_pct:.4f}%)")
    print(f"   最便宜: {min(working, key=working.get)}")
    print(f"   最贵:   {max(working, key=working.get)}")
else:
    print("⚠️ 只有一个交易所可用，无法对比价差")

✅ coinbase   → $60,119.22
✅ kraken     → $60,064.50
✅ gemini     → $60,070.42

━━━━━━━━━━━━━━━━━━━━━━
📏 最大价差: $54.72 (0.0911%)
   最便宜: kraken
   最贵:   coinbase


In [6]:
# 用 CoinGecko 拉全市场数据（不需要 CCXT）
import requests

symbols = ['bitcoin', 'ethereum', 'solana', 'ripple', 'cardano']
data = []

for coin_id in symbols:
    url = f"https://api.coingecko.com/api/v3/simple/price?ids={coin_id}&vs_currencies=usd&include_24hr_vol=true&include_market_cap=true"
    r = requests.get(url).json()

    if coin_id in r:
        price = r[coin_id]['usd']
        vol = r[coin_id]['usd_24h_vol']
        mcap = r[coin_id]['usd_market_cap']

        # OI 数据 CoinGecko 没有，用 Coinalyze 手动查
        # 这里用市值 × 估的 OI/市值比（参考上次聊的：BTC~1.7%, ETH~5%, SOL~2.7%）
        oi_ratios = {
            'bitcoin': 0.017,
            'ethereum': 0.050,
            'solana': 0.027,
            'ripple': 0.020,
            'cardano': 0.015
        }
        estimated_oi = mcap * oi_ratios.get(coin_id, 0.02)
        oi_vol_ratio = estimated_oi / vol if vol > 0 else 0

        data.append({
            '币种': coin_id[:4].upper(),
            '价格': f"${price:,.0f}",
            '流通市值': f"${mcap/1e9:.1f}B",
            '24h成交量': f"${vol/1e9:.1f}B",
            '预估OI': f"${estimated_oi/1e9:.2f}B",
            'OI/Volume': f"{oi_vol_ratio:.2f}",
            '评级': '🔴高危' if oi_vol_ratio > 2 else '🟡警戒' if oi_vol_ratio > 1 else '🟢健康'
        })

df = pd.DataFrame(data)
print("📊 预估 OI/Volume 分析")
print("━━━━━━━━━━━━━━━━━━━━━━━━━━")
print(df.to_string(index=False))
print()
print("💡 提示: OI/Volume > 1 = 每1块钱交易有>1块钱未平仓，杠杆密集")
print("💡 提示: OI/Volume > 2 = 高杠杆区，一个方向爆仓可能连锁反应")

📊 预估 OI/Volume 分析
━━━━━━━━━━━━━━━━━━━━━━━━━━
  币种      价格     流通市值 24h成交量    预估OI OI/Volume  评级
BITC $59,990 $1202.5B $20.4B $20.44B      1.00 🟡警戒
ETHE  $1,579  $190.5B  $7.4B  $9.53B      1.28 🟡警戒
SOLA     $72   $42.0B  $2.4B  $1.13B      0.48 🟢健康
RIPP      $1   $65.2B  $1.4B  $1.30B      0.96 🟢健康
CARD      $0    $5.4B  $0.2B  $0.08B      0.33 🟢健康

💡 提示: OI/Volume > 1 = 每1块钱交易有>1块钱未平仓，杠杆密集
💡 提示: OI/Volume > 2 = 高杠杆区，一个方向爆仓可能连锁反应
